### In this section I aim to optimise our model 1 decision tree regressor by first doing feature selection, then doing some MSE train and test error to look at the bias vs. variance tradeoff. In the end I will do some gridsearch to find the optimal model.

In [34]:
# Importing necessary class' and functions
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from M1 import rss, Node, build_tree, best_split, predict_one, predict

In [ ]:
# Loading in the dataset
df = pd.read_csv("../data/Cleaned_data_train.csv")

# Removing the areas as they are directly correlated with density as stated in the description of the data
df = df.drop(labels=["Area_A", "Area_B", "Area_C", "Area_D", "Area_E", "Area_F"], axis = 1)

# Removing exposure above 1 as an exposure period can't be more than 1 (1 year), so it is seen as incomplete data.
df = df[df["Exposure"] <= 1]
df

,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,Density,VehBrand_B1,VehBrand_B10,VehBrand_B11,...,Region_R53,Region_R54,Region_R72,Region_R73,Region_R74,Region_R82,Region_R83,Region_R91,Region_R93,Region_R94
0,0,0.43,7,18,36,95,1054,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0.10,7,17,80,95,598,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0.33,7,3,36,76,4172,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0.56,5,4,73,52,15,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0.27,8,0,37,50,3021,0,0,1,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542405,0,0.20,6,10,32,76,1314,0,0,0,...,0,0,0,0,0,0,0,0,0,0
542406,0,0.06,10,14,34,60,685,0,0,0,...,0,0,1,0,0,0,0,0,0,0
542407,0,0.34,6,8,32,95,242,0,0,0,...,0,0,0,0,0,1,0,0,0,0
542408,0,0.72,9,7,39,72,3301,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [ ]:
X = df.drop("ClaimNb", axis=1)
y = df['ClaimNb']

X = X.to_numpy()
y = y.to_numpy()

MSE: 0.05436604580158044
R² Score: 0.0242


In [ ]:
def stratified_bootstrap_indices(y, n_samples, min_positive_frac=0.05, random_state=None):
    rng = np.random.default_rng(random_state)

    pos_idx = np.where(y > 0)[0]
    neg_idx = np.where(y == 0)[0]

    n_pos = max(1, int(n_samples * min_positive_frac))
    n_neg = n_samples - n_pos

    pos_sample = rng.choice(pos_idx, size=n_pos, replace=True)
    neg_sample = rng.choice(neg_idx, size=n_neg, replace=True)

    return np.concatenate([pos_sample, neg_sample])



In [36]:
def permutation_importance_custom(
    tree,
    X,
    y,
    n_repeats=5,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    baseline_pred = predict(tree, X)
    baseline_loss = mean_squared_error(y, baseline_pred)

    n_features = X.shape[1]
    importances = np.zeros(n_features)

    for j in range(n_features):
        losses = []

        for _ in range(n_repeats):
            X_permuted = X.copy()
            rng.shuffle(X_permuted[:, j])

            perm_pred = predict(tree, X_permuted)
            losses.append(mean_squared_error(y, perm_pred))

        importances[j] = np.mean(losses) - baseline_loss

    return importances


In [30]:
def bagged_permutation_importance_custom(
    X_train, y_train,
    X_val, y_val,
    n_bags=20,
    n_repeats=5,
    min_positive_frac=0.2,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    n_features = X_train.shape[1]

    all_importances = []

    for b in range(n_bags):
        boot_idx = stratified_bootstrap_indices(
            y_train,
            n_samples=len(y_train),
            min_positive_frac=min_positive_frac,
            random_state=random_state + b
        )

        tree = build_tree(X_train[boot_idx], y_train[boot_idx])

        importances = permutation_importance_custom(
            tree,
            X_val,
            y_val,
            n_repeats=n_repeats,
            random_state=random_state + b
        )

        all_importances.append(importances)

    return np.mean(all_importances, axis=0), np.std(all_importances, axis=0)


In [31]:
from sklearn.model_selection import KFold

def cv_bagged_permutation_importance_custom(
    X, y,
    n_splits=5,
    n_bags=20,
    n_repeats=5,
    min_positive_frac=0.2
):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_importances = []

    for train_idx, val_idx in kf.split(X):
        mean_imp, _ = bagged_permutation_importance_custom(
            X[train_idx], y[train_idx],
            X[val_idx], y[val_idx],
            n_bags=n_bags,
            n_repeats=n_repeats,
            min_positive_frac=min_positive_frac
        )
        fold_importances.append(mean_imp)

    fold_importances = np.array(fold_importances)
    return fold_importances.mean(axis=0), fold_importances.std(axis=0)


In [37]:
mean_imp, std_imp = cv_bagged_permutation_importance_custom(X, y)

selected_features = np.where(mean_imp > std_imp)[0]


KeyboardInterrupt: 